In [5]:
import os
import io
import pandas as pd
import numpy as np
import statsmodels.api as sm
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

# ----------------------------
# 1. Miljøvariabler
# ----------------------------
load_dotenv(".env")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "processeddata"

# ----------------------------
# 2. LES FORBRUK
# ----------------------------
blob_client1 = blob_service_client.get_blob_client(
    container=container_name,
    blob="BREIVE_processed.csv"
)
data1 = blob_client1.download_blob().readall()
df_consumption = pd.read_csv(io.BytesIO(data1))

# ----------------------------
# 3. LES VÆR
# ----------------------------
blob_client2 = blob_service_client.get_blob_client(
    container=container_name,
    blob="BREIVE_weather.csv"
)
data2 = blob_client2.download_blob().readall()
df_weather = pd.read_csv(io.BytesIO(data2))

print("Begge filer lastet!")

# ----------------------------
# 4. Fix timestamp
# ----------------------------
df_consumption['timestamp'] = pd.to_datetime(df_consumption['timestamp'])
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])

# ----------------------------
# 5. Merge data
# ----------------------------
df = pd.merge(df_consumption, df_weather, on="timestamp", how="inner")

print("Antall rader etter merge:", len(df))

# ----------------------------
# 6. Velg variabler
# ----------------------------
feature_cols = [
    "hour",
    "weekday",
    "month",
    "is_weekend",
    "is_holiday",
    "air_temperature",
    "wind_speed",
    "precipitation_mm",
    "norgespris"   # 0 = før, 1 = etter
]

target_col = "value_kwh"

# ----------------------------
# 7. Lag modell-datasett
# ----------------------------
df_model = df[feature_cols + [target_col]].copy()

# 🔥 FIX: bool → int
for col in df_model.columns:
    if df_model[col].dtype == "bool":
        df_model[col] = df_model[col].astype(int)

# 🔥 FIX: alt → float
df_model = df_model.astype(float)

# 🔥 Fjern NaN
df_model = df_model.dropna()

print("Rader brukt i modell:", len(df_model))

# ----------------------------
# 8. Definer X og y
# ----------------------------
X = df_model[feature_cols]
y = df_model[target_col]

# Legg til konstant (intercept)
X = sm.add_constant(X)

# Debug (kan fjernes senere)
print("\nDatatyper:")
print(X.dtypes)

# ----------------------------
# 9. Kjør regresjon
# ----------------------------
model = sm.OLS(y, X).fit()

# ----------------------------
# 10. Resultater
# ----------------------------
print("\n=== REGRESJONSRESULTATER ===")
print(model.summary().as_text())

Begge filer lastet!
Antall rader etter merge: 5479690
Rader brukt i modell: 5479690

Datatyper:
const               float64
hour                float64
weekday             float64
month               float64
is_weekend          float64
is_holiday          float64
air_temperature     float64
wind_speed          float64
precipitation_mm    float64
norgespris          float64
dtype: object

=== REGRESJONSRESULTATER ===
                            OLS Regression Results                            
Dep. Variable:              value_kwh   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                 2.930e+04
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:38:31   Log-Likelihood:            -1.1673e+07
No. Observations:             5479690   AIC:                         2.335e+07
Df Residuals:              